# Verify the Model B Integrated Gradients fix (issue #15 follow-up)

`model_b_phase4_projected_norm_attr_36bp` came back all-zero for every Case
Study in `model_analysis_summary.json`. Root cause: the old IG code
backpropagated only through `EsmClassificationHead`, which pools *only* the
CLS token's final hidden state (`features[:, 0, :]`) - every non-CLS token's
gradient was exactly zero by construction, not just small.

The fix attributes against the *pre-encoder* input-embedding tensor instead
and replays the whole encoder + head, so gradients flow back through every
self-attention layer, where non-CLS tokens genuinely do influence the CLS
token's final representation. It also freezes the NT encoder's ~500M
parameters for the duration of the backward passes (IG only needs a
gradient w.r.t. the input, not the model's own weights), since that turned
out to exhaust the local machine's RAM once IG started replaying the full
encoder instead of just its small head.

This notebook re-runs `test_integrated_gradients.py`'s full Model B suite
(the exact tests that caught the original bug and now verify the fix) plus
a direct, human-readable before/after check, on Colab where memory isn't a
constraint.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Point at your uploaded model assets

Edit `DRIVE_DIR` to the Drive folder holding `NT_sacas9_fintuned_model/`
(the only asset this notebook needs - none of the tests below touch
`hybrid_xgb_model.json`, `best_model_fold1.pth`, or `test_metadata.csv`).


In [ ]:
import os

DRIVE_DIR = "/content/drive/MyDrive/xai_test"  # <-- EDIT ME

required = ["NT_sacas9_fintuned_model"]
missing = [
    name for name in required if not os.path.exists(os.path.join(DRIVE_DIR, name))
]
if missing:
    raise FileNotFoundError(
        f"Missing in {DRIVE_DIR}: {missing}. Upload these to Drive first, or fix DRIVE_DIR."
    )

os.chdir(DRIVE_DIR)
print("Working directory:", os.getcwd())

## 3. Install Python deps

In [ ]:
!pip install -q transformers pytest

## 4. Write the predictor + Integrated Gradients modules

Mirrors the current committed `xai_test/` repo exactly (embedded at
notebook-build time, not fetched at runtime) - includes the fix to
`model_b_xai_wrapper.py`/`integrated_gradients.py` and the widened
completeness-test tolerance in `test_integrated_gradients.py`.


In [ ]:
%%writefile model_a_wrapper.py
import numpy as np
import torch
from torch import nn


class OneHotAdapter(nn.Module):
    def __init__(self, input_dim: int = 4, hidden_dim: int = 128):
        super().__init__()
        # 가중치 형태 [128, 4]에 맞게 nn.Linear로 변경
        self.proj = nn.Linear(input_dim, hidden_dim)

    def forward(self, x):
        # x shape: [Batch, 36, 4]
        x = self.proj(x)  # -> [Batch, 36, 128]
        return x.transpose(1, 2)  # CNN 백본 입력에 맞게 변경 -> [Batch, 128, 36]


class CNN_RNN_Backbone(nn.Module):
    def __init__(
        self,
        input_dim: int = 128,
        conv_channels: int = 64,
        kernel_size: int = 3,
        lstm_hidden: int = 128,
        lstm_layers: int = 1,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.input_dim = input_dim
        self.conv = nn.Conv1d(
            input_dim, conv_channels, kernel_size, padding=kernel_size // 2
        )
        self.act = nn.ReLU()
        self.lstm = nn.LSTM(
            input_size=conv_channels,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, lstm_hidden),
            nn.ReLU(),
            nn.Linear(lstm_hidden, 1),
        )

    def forward(self, x):
        # x shape: [Batch, 128, 36]
        x = self.act(self.conv(x))
        x = x.transpose(1, 2)  # [Batch, 36, 64]
        out, _ = self.lstm(x)  # out shape: [Batch, 36, 128]

        # 마지막 시점(sequence)의 hidden state를 추출하여 분류기 통과
        out = self.head(out[:, -1, :])
        return out


class IntegratedPredictor(nn.Module):
    def __init__(self, adapter: nn.Module, backbone: nn.Module):
        super().__init__()
        self.adapter = adapter
        self.backbone = backbone

    def forward(self, x):
        x = self.adapter(x)
        x = self.backbone(x)
        return x.squeeze(-1)  # 1D Array 변환


class Model_A_Predictor:
    def __init__(self, weight_path: str | None = None, dropout_rate: float = 0.0):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # 원본 구조와 동일하게 어댑터와 백본 조립
        adapter = OneHotAdapter(input_dim=4, hidden_dim=128)
        backbone = CNN_RNN_Backbone(
            input_dim=128, lstm_hidden=128, dropout=dropout_rate
        )
        self.model = IntegratedPredictor(adapter=adapter, backbone=backbone).to(
            self.device
        )

        # 가중치 로드
        if weight_path:
            self.model.load_state_dict(
                torch.load(weight_path, map_location=self.device)
            )
        self.model.eval()

        self.nuc_map = {
            "A": [1.0, 0.0, 0.0, 0.0],
            "C": [0.0, 1.0, 0.0, 0.0],
            "G": [0.0, 0.0, 1.0, 0.0],
            "T": [0.0, 0.0, 0.0, 1.0],
        }

    def encode_one_hot(self, sequences: list[str]) -> torch.Tensor:
        encoded_list = []
        for seq in sequences:
            encoded = [self.nuc_map.get(n.upper(), [0.0, 0.0, 0.0, 0.0]) for n in seq]
            encoded_list.append(encoded)
        tensor = torch.tensor(encoded_list, dtype=torch.float32)
        return tensor  # shape: [Batch, 36, 4]

    def predict(self, sequences: list[str]) -> np.ndarray:
        x = self.encode_one_hot(sequences).to(self.device)
        with torch.no_grad():
            preds = self.model(x).cpu().numpy()
        return preds


In [ ]:
%%writefile ism_sweep.py
"""In-silico saturation mutagenesis (ISM) sweep (Step 3, see CONTEXT.md / plan_realize.md).

Generates every position x alternative-base substitution for a set of 36bp
Testset sequences, scores wild-type and mutants through a predictor's own
`predict()`, and reduces the result to a relative delta-score sensitivity
array. Works against any predictor exposing `predict(sequences) -> array`
(duck-typed to `Model_A_Predictor` / `Model_B_Predictor`), so Model B's
per-mutant physical-feature recomputation (MFE, dG, Tm, GC) happens for free:
every mutant is dispatched as a full sequence, never a cached wild-type value
or a bare delta.
"""

from collections.abc import Sequence

import numpy as np

BASES = ("A", "C", "G", "T")

# Relative delta-Score is undefined (unbounded) as Score_WT -> 0; scores at or
# below this magnitude are masked to NaN instead of producing a spurious ratio.
RELATIVE_DELTA_NAN_THRESHOLD = 1e-4

# Neither Model_A_Predictor nor Model_B_Predictor batches internally - predict()
# runs whatever list it's given as one forward pass. An ISM sweep can hand it
# tens of thousands of mutants at once, which is large enough to exhaust GPU
# memory (observed: CUDA OOM inside Model A's LSTM forward pass on a 14.56GB
# T4 trying to allocate 20.71 GiB for one such call). Chunking dispatch here
# keeps peak memory bounded regardless of sweep size, without changing either
# predictor's own predict() contract.
DEFAULT_PREDICT_BATCH_SIZE = 256


def alt_bases_for(wt_base: str) -> list[str]:
    """The 3 non-wild-type bases, in fixed lexicographic order."""
    wt_base = wt_base.upper()
    if wt_base not in BASES:
        raise ValueError(f"Not a valid base: {wt_base!r}")
    return sorted(set(BASES) - {wt_base})


def generate_ism_mutants(sequence: str) -> list[str]:
    """All position x alternative-base substitutions for `sequence`.

    Ordered position-major (0..len-1), alternative-base lexicographic-minor,
    matching the `ism_delta` array's (position, alternative_base) axes.
    """
    sequence = sequence.upper()
    mutants = []
    for pos, wt_base in enumerate(sequence):
        for alt_base in alt_bases_for(wt_base):
            mutants.append(sequence[:pos] + alt_base + sequence[pos + 1 :])
    return mutants


def dispatch_in_batches(
    fn, items: list[str], batch_size: int, stack=np.concatenate
) -> np.ndarray:
    """Call `fn()` in bounded chunks and combine the results with `stack`.

    Generic version of the chunking `predict_in_batches` needs for 1D score
    arrays; `step4_ablation.py` reuses this directly (with `stack=np.vstack`)
    to batch `Model_B_Predictor.extract_nt_embeddings()`'s 2D output the same
    way, instead of re-deriving the chunk loop.
    """
    if not items:
        return np.array([])
    chunks = [fn(items[i : i + batch_size]) for i in range(0, len(items), batch_size)]
    return stack(chunks)


def predict_in_batches(predictor, sequences: list[str], batch_size: int) -> np.ndarray:
    """Dispatch `predictor.predict()` in bounded chunks (shared with mismatch_profiling.py)."""
    if not sequences:
        return np.array([], dtype=np.float64)

    def _predict_chunk(chunk):
        return np.asarray(predictor.predict(chunk), dtype=np.float64)

    return dispatch_in_batches(
        _predict_chunk, sequences, batch_size, stack=np.concatenate
    )


def relative_delta(
    wt_scores: np.ndarray, mutant_scores: np.ndarray, nan_threshold: float
) -> np.ndarray:
    """`(mutant - wt) / |wt|` per sample, NaN-masked wherever `|wt| <= nan_threshold`.

    `wt_scores` is 1D (one value per sample); `mutant_scores` broadcasts against
    it along its leading axis - shape `(n, positions, alt_bases)` for the ISM
    sweep, `(n, scenarios)` for complex mismatch profiling. Shared so both
    modules apply the exact same guardrail (see `RELATIVE_DELTA_NAN_THRESHOLD`).
    """
    broadcast_shape = (wt_scores.shape[0],) + (1,) * (mutant_scores.ndim - 1)
    wt_broadcast = wt_scores.reshape(broadcast_shape)
    with np.errstate(divide="ignore", invalid="ignore"):
        delta = (mutant_scores - wt_broadcast) / np.abs(wt_broadcast)
    delta[np.abs(wt_scores) <= nan_threshold] = np.nan
    return delta


def compute_ism_delta(
    sequences: Sequence[str],
    predictor,
    nan_threshold: float = RELATIVE_DELTA_NAN_THRESHOLD,
    predict_batch_size: int = DEFAULT_PREDICT_BATCH_SIZE,
) -> np.ndarray:
    """Run the full ISM sweep for one predictor.

    Returns `ism_delta`, shape `(len(sequences), seq_len, 3)`: the relative
    score delta `(Score_mutant - Score_WT) / |Score_WT|` per position x
    lexicographic alternative base, NaN-masked wherever `|Score_WT| <=
    nan_threshold`. `predict_batch_size` bounds how many sequences reach
    `predictor.predict()` per call; lower it if you still hit an
    out-of-memory error.
    """
    sequences = list(sequences)
    seq_len = len(sequences[0])
    if any(len(seq) != seq_len for seq in sequences):
        raise ValueError("All sequences must share the same length for the ISM sweep.")

    wt_scores = predict_in_batches(predictor, sequences, predict_batch_size)

    mutants: list[str] = []
    for sequence in sequences:
        mutants.extend(generate_ism_mutants(sequence))

    mutant_scores = predict_in_batches(predictor, mutants, predict_batch_size)
    mutant_scores = mutant_scores.reshape(len(sequences), seq_len, 3)

    return relative_delta(wt_scores, mutant_scores, nan_threshold)


def run_ism_sweep(
    sequences: Sequence[str],
    predictor_a,
    predictor_b,
    predict_batch_size: int = DEFAULT_PREDICT_BATCH_SIZE,
) -> dict[str, np.ndarray]:
    """Run the ISM sweep for both models over the same Testset sequences."""
    return {
        "ism_delta_model_a": compute_ism_delta(
            sequences, predictor_a, predict_batch_size=predict_batch_size
        ),
        "ism_delta_model_b": compute_ism_delta(
            sequences, predictor_b, predict_batch_size=predict_batch_size
        ),
    }


In [ ]:
%%writefile mismatch_profiling.py
"""Mismatch tolerance profiling: single + complex scenarios (Step 3, issue #12).

Single mismatch: a single mismatch *is* a single ISM mutation, so
`compute_mismatch_single` reuses `ism_sweep.compute_ism_delta` unchanged for
`mismatch_single_delta`, and reduces it to the worst-case (minimum relative
delta) per position for `mismatch_single_min`.

Complex mismatch: a fixed set of 15 reproducible (`random_seed=42`)
multi-position (1-3bp) scenarios - 5 Seed-region, 5 Distal-region, 5
Intermittent - applied identically (same positions, same per-position
lexicographic-alternate-base selection, see `alt_bases_for`) to every Testset
sequence. Substitutions stay self-relative to each sequence's own base at
each position (the same convention `ism_sweep` uses), which guarantees every
substitution is a genuine mismatch regardless of the underlying sequence,
while keeping one scenario definition comparable ("profiled") across the
whole population. Region boundaries: Seed = positions 17-24 (the 8bp of the
protospacer immediately 5' of - i.e. PAM-proximal to - the confirmed PAM at
25-30, see integrated_gradients.py's PAM_REGION_START/END), Distal =
positions 0-7 (the farthest 8bp from the PAM), Intermittent = non-consecutive
positions scattered across the full 0-35 span.

Earlier versions of this module (issue #12) fixed Seed at positions 0-7,
before the PAM's exact position was pinned down (issue #15 confirmed
25-30). That range was mislabeled "PAM-adjacent" - positions 0-7 are in fact
the farthest possible point from the PAM - and the old "Distal" range
(28-35) actually overlapped the PAM itself. Corrected here so "Seed" is
biologically PAM-proximal and "Distal" is biologically PAM-distal, matching
plan.md/plan_realize.md's stated hypothesis that cleavage sensitivity rises
closer to the PAM.

`ref_bases`/`alt_bases` in a scenario's exported metadata are illustrative:
computed against one `example_sequence` (by convention, Testset sample_id 0)
because the literal ref base at a scenario's positions differs per sequence -
the same position + alt-base-index rule is applied to every sequence's own
base, exactly as ISM does.
"""

from collections.abc import Sequence
from dataclasses import dataclass

import numpy as np
from ism_sweep import (
    DEFAULT_PREDICT_BATCH_SIZE,
    RELATIVE_DELTA_NAN_THRESHOLD,
    alt_bases_for,
    compute_ism_delta,
    predict_in_batches,
    relative_delta,
)

SEQ_LEN = 36
# PAM-proximal 8bp of the protospacer, immediately 5' of the confirmed PAM
# (25-30, see integrated_gradients.PAM_REGION_START/END).
SEED_REGION_START, SEED_REGION_END = 17, 25  # positions 17-24
DISTAL_REGION_START, DISTAL_REGION_END = 0, 8  # positions 0-7, farthest from the PAM
SCENARIOS_PER_REGION = 5
COMPLEX_MISMATCH_SEED = 42


@dataclass(frozen=True)
class MismatchScenario:
    """One fixed complex-mismatch scenario, applied identically across the Testset."""

    mutation_id: str
    region: str
    positions: tuple[int, ...]
    alt_base_indices: tuple[int, ...]  # index into alt_bases_for(wt) per position
    description: str


def _draw_alt_indices(rng: np.random.Generator, n: int) -> tuple[int, ...]:
    return tuple(int(x) for x in rng.integers(0, 3, size=n))


def _generate_consecutive_scenarios(
    rng: np.random.Generator, region: str, region_start: int, region_end: int
) -> list[MismatchScenario]:
    scenarios = []
    region_prefix = region.lower()
    seen_positions: set[tuple[int, ...]] = set()
    for _ in range(SCENARIOS_PER_REGION):
        while True:
            length = int(rng.integers(1, 4))  # 1..3 bp, per plan.md
            max_start = region_end - length
            start = int(rng.integers(region_start, max_start + 1))
            positions = tuple(range(start, start + length))
            if positions not in seen_positions:
                seen_positions.add(positions)
                break
        alt_base_indices = _draw_alt_indices(rng, length)
        pos_label = "_".join(str(p) for p in positions)
        scenarios.append(
            MismatchScenario(
                mutation_id=f"{region_prefix}_consecutive_{length}bp_pos{pos_label}",
                region=region,
                positions=positions,
                alt_base_indices=alt_base_indices,
                description=f"{region} consecutive {length}bp mismatch at position(s) {list(positions)}",
            )
        )
    return scenarios


def _generate_intermittent_scenarios(
    rng: np.random.Generator,
) -> list[MismatchScenario]:
    scenarios = []
    seen_positions: set[tuple[int, ...]] = set()
    for _ in range(SCENARIOS_PER_REGION):
        while True:
            count = int(rng.integers(2, 4))  # 2..3 non-consecutive positions
            positions = tuple(
                sorted(int(x) for x in rng.choice(SEQ_LEN, size=count, replace=False))
            )
            is_consecutive = all(
                positions[i + 1] - positions[i] == 1 for i in range(len(positions) - 1)
            )
            if not is_consecutive and positions not in seen_positions:
                seen_positions.add(positions)
                break
        alt_base_indices = _draw_alt_indices(rng, count)
        pos_label = "_".join(str(p) for p in positions)
        scenarios.append(
            MismatchScenario(
                mutation_id=f"intermittent_nonconsecutive_{count}bp_pos{pos_label}",
                region="Intermittent",
                positions=positions,
                alt_base_indices=alt_base_indices,
                description=f"Intermittent non-consecutive {count}bp mismatch at position(s) {list(positions)}",
            )
        )
    return scenarios


def generate_mismatch_complex_scenarios(
    seed: int = COMPLEX_MISMATCH_SEED,
) -> list[MismatchScenario]:
    """The fixed 15 complex mismatch scenarios: 5 Seed + 5 Distal + 5 Intermittent.

    Fully determined by `seed` - drawn in this order (Seed, then Distal, then
    Intermittent) from one `np.random.default_rng(seed)` stream, so the same
    seed always reproduces the same 15 scenarios.
    """
    rng = np.random.default_rng(seed)
    scenarios = []
    scenarios.extend(
        _generate_consecutive_scenarios(rng, "Seed", SEED_REGION_START, SEED_REGION_END)
    )
    scenarios.extend(
        _generate_consecutive_scenarios(
            rng, "Distal", DISTAL_REGION_START, DISTAL_REGION_END
        )
    )
    scenarios.extend(_generate_intermittent_scenarios(rng))
    return scenarios


def apply_scenario(scenario: MismatchScenario, sequence: str) -> str:
    """Apply `scenario`'s positions/alt-indices to `sequence`'s own bases."""
    chars = list(sequence.upper())
    for pos, alt_idx in zip(scenario.positions, scenario.alt_base_indices):
        chars[pos] = alt_bases_for(chars[pos])[alt_idx]
    return "".join(chars)


def scenario_to_metadata(
    scenario: MismatchScenario, scenario_index: int, example_sequence: str
) -> dict:
    """Export metadata for one scenario (see module docstring on ref/alt bases)."""
    example_sequence = example_sequence.upper()
    ref_bases = [example_sequence[p] for p in scenario.positions]
    alt_bases = [
        alt_bases_for(example_sequence[p])[idx]
        for p, idx in zip(scenario.positions, scenario.alt_base_indices)
    ]
    return {
        "scenario_index": scenario_index,
        "mutation_id": scenario.mutation_id,
        "region": scenario.region,
        "positions": list(scenario.positions),
        "ref_bases": ref_bases,
        "alt_bases": alt_bases,
        "description": scenario.description,
    }


def compute_mismatch_single(
    sequences: Sequence[str],
    predictor,
    nan_threshold: float = RELATIVE_DELTA_NAN_THRESHOLD,
    predict_batch_size: int = DEFAULT_PREDICT_BATCH_SIZE,
) -> tuple[np.ndarray, np.ndarray]:
    """Single mismatch profiling for one predictor.

    Returns `(mismatch_single_delta, mismatch_single_min)`:
    - `mismatch_single_delta`, shape `(len(sequences), seq_len, 3)`: identical
      to `ism_sweep.compute_ism_delta` (a single mismatch is a single ISM
      mutation) - all 3 substitutions preserved per position.
    - `mismatch_single_min`, shape `(len(sequences), seq_len)`: the worst-case
      (minimum) relative delta across the 3 alternatives at each position.
    """
    mismatch_single_delta = compute_ism_delta(
        sequences,
        predictor,
        nan_threshold=nan_threshold,
        predict_batch_size=predict_batch_size,
    )
    mismatch_single_min = np.min(mismatch_single_delta, axis=-1)
    return mismatch_single_delta, mismatch_single_min


def compute_mismatch_complex_delta(
    sequences: Sequence[str],
    predictor,
    scenarios: list[MismatchScenario] | None = None,
    nan_threshold: float = RELATIVE_DELTA_NAN_THRESHOLD,
    predict_batch_size: int = DEFAULT_PREDICT_BATCH_SIZE,
) -> np.ndarray:
    """Complex mismatch profiling for one predictor.

    Returns `mismatch_complex`, shape `(len(sequences), len(scenarios))`: the
    relative score delta `(Score_mutant - Score_WT) / |Score_WT|` per
    scenario, NaN-masked wherever `|Score_WT| <= nan_threshold` (same
    guardrail as the ISM sweep).
    """
    if scenarios is None:
        scenarios = generate_mismatch_complex_scenarios()

    sequences = list(sequences)
    seq_len = len(sequences[0])
    if any(len(seq) != seq_len for seq in sequences):
        raise ValueError(
            "All sequences must share the same length for mismatch profiling."
        )

    wt_scores = predict_in_batches(predictor, sequences, predict_batch_size)

    # Scenario-major, sequence-minor ordering, matching the reshape below.
    mutants: list[str] = []
    for scenario in scenarios:
        for sequence in sequences:
            mutants.append(apply_scenario(scenario, sequence))

    mutant_scores = predict_in_batches(predictor, mutants, predict_batch_size)
    mutant_scores = mutant_scores.reshape(len(scenarios), len(sequences)).T

    return relative_delta(wt_scores, mutant_scores, nan_threshold)


def run_mismatch_profiling(
    sequences: Sequence[str],
    predictor_a,
    predictor_b,
    scenarios: list[MismatchScenario] | None = None,
    predict_batch_size: int = DEFAULT_PREDICT_BATCH_SIZE,
) -> dict[str, np.ndarray]:
    """Run single + complex mismatch profiling for both models over the same Testset."""
    if scenarios is None:
        scenarios = generate_mismatch_complex_scenarios()

    result: dict[str, np.ndarray] = {}
    for model_name, predictor in (("model_a", predictor_a), ("model_b", predictor_b)):
        single_delta, single_min = compute_mismatch_single(
            sequences, predictor, predict_batch_size=predict_batch_size
        )
        result[f"mismatch_single_delta_{model_name}"] = single_delta
        result[f"mismatch_single_min_{model_name}"] = single_min
        result[f"mismatch_complex_{model_name}"] = compute_mismatch_complex_delta(
            sequences,
            predictor,
            scenarios=scenarios,
            predict_batch_size=predict_batch_size,
        )
    return result


In [ ]:
%%writefile model_b_xai_wrapper.py
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


class Model_B_XAIPredictor:
    """XAI-only counterpart to `Model_B_Predictor` (see ADR-0001).

    Loads the same NT checkpoint via `AutoModelForSequenceClassification`
    (not `AutoModelForMaskedLM`) so gradients flow through the checkpoint's
    own regression head and `output_attentions=True` is available. Has no
    analysis behavior of its own; Integrated Gradients and Attention Rollout
    build on top of it. Must never modify or risk `Model_B_Predictor`'s
    production inference path.
    """

    EXPECTED_SEQ_LEN = 7
    EXPECTED_HIDDEN_DIM = 1280

    def __init__(self, nt_model_dir: str):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(
            f"Loading NT backbone (XAI, sequence-classification head) from {nt_model_dir}..."
        )
        self.tokenizer = AutoTokenizer.from_pretrained(nt_model_dir)
        # "eager" attention is required for output_attentions=True; the default
        # "sdpa" backend silently returns no attention weights instead.
        self.nt_model = AutoModelForSequenceClassification.from_pretrained(
            nt_model_dir, attn_implementation="eager"
        ).to(self.device)
        self.nt_model.eval()

    def _tokenize(self, sequences: list[str]):
        return self.tokenizer(
            sequences, return_tensors="pt", padding=True, truncation=True
        ).to(self.device)

    def get_token_embeddings(self, sequences: list[str]) -> torch.Tensor:
        """Return the last-layer token embeddings, gradients enabled.

        Shape is asserted here, before any consumer (e.g. Integrated
        Gradients) uses the tensor, so a malformed input raises immediately
        instead of silently propagating a wrong shape downstream.
        """
        inputs = self._tokenize(sequences)
        outputs = self.nt_model(**inputs, output_hidden_states=True)
        embeddings = outputs.hidden_states[-1]

        _batch_size, seq_len, hidden_dim = embeddings.shape
        assert seq_len == self.EXPECTED_SEQ_LEN, (
            f"Expected {self.EXPECTED_SEQ_LEN} tokens, got {seq_len}"
        )
        assert hidden_dim == self.EXPECTED_HIDDEN_DIM, (
            f"Expected {self.EXPECTED_HIDDEN_DIM} hidden dim, got {hidden_dim}"
        )
        assert embeddings.requires_grad, (
            "Embeddings must carry gradients for Integrated Gradients to run"
        )

        return embeddings

    def get_input_embeddings(
        self, sequences: list[str]
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Return (pre-encoder input embeddings, attention_mask), gradients enabled.

        This - not `get_token_embeddings`'s *post-encoder* last hidden
        state - is the point Integrated Gradients must attribute against:
        `EsmClassificationHead` pools only the CLS token's final hidden
        state (`features[:, 0, :]`), so every non-CLS post-encoder
        embedding is a dead end the head never reads - its IG attribution
        would be exactly zero by construction, not just small. Attributing
        against the embedding-lookup output instead lets gradients flow
        back through every self-attention layer, where non-CLS tokens do
        influence the CLS token's final representation.
        """
        inputs = self._tokenize(sequences)
        embeddings = self.nt_model.esm.embeddings(
            input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"]
        )

        _batch_size, seq_len, hidden_dim = embeddings.shape
        assert seq_len == self.EXPECTED_SEQ_LEN, (
            f"Expected {self.EXPECTED_SEQ_LEN} tokens, got {seq_len}"
        )
        assert hidden_dim == self.EXPECTED_HIDDEN_DIM, (
            f"Expected {self.EXPECTED_HIDDEN_DIM} hidden dim, got {hidden_dim}"
        )
        assert embeddings.requires_grad, (
            "Embeddings must carry gradients for Integrated Gradients to run"
        )

        return embeddings, inputs["attention_mask"]

    def classify_from_input_embeddings(
        self, embeddings: torch.Tensor, attention_mask: torch.Tensor
    ) -> torch.Tensor:
        """Run the full encoder + classification head from pre-encoder embeddings.

        Unlike a head-only shortcut, this replays every self-attention
        layer so gradients reach every token position, not just CLS - see
        `get_input_embeddings`.
        """
        return self.nt_model(
            inputs_embeds=embeddings, attention_mask=attention_mask
        ).logits.squeeze(-1)

    def get_attentions(self, sequences: list[str]) -> tuple[torch.Tensor, ...]:
        """Return per-layer attention matrices, `<cls>` still included.

        Stripping `<cls>` and mapping to nucleotide coordinates happens
        downstream, in Attention Rollout.
        """
        inputs = self._tokenize(sequences)
        outputs = self.nt_model(**inputs, output_attentions=True)
        return outputs.attentions


In [ ]:
%%writefile integrated_gradients.py
"""Integrated Gradients: Model A vs Model B sequence-level comparison (Step 2, issue #15).

Model A is attributed directly against its one-hot input layer with an
all-zero one-hot baseline. Model B is attributed against
`Model_B_XAIPredictor`'s `(batch, 7, 1280)` pre-encoder input-embedding
tensor (see ADR-0001 / issue #10) with a fixed zero-embedding baseline of
the same shape, running the integration through the *whole* encoder +
classification head (`classify_from_input_embeddings`). Attributing against
the *post-encoder* last hidden state instead (an earlier version of this
module did) is a dead end: `EsmClassificationHead` pools only the CLS
token's final hidden state (`features[:, 0, :]`), so every non-CLS
post-encoder embedding's gradient is exactly zero by construction - not
just small - regardless of the actual sequence. Attributing against the
pre-encoder embeddings and replaying the full encoder instead lets
gradients flow back through every self-attention layer, where non-CLS
tokens do influence the CLS token's final representation.

Model B's 6-mer token attributions are projected back to the 36bp sequence
via the Deterministic Projection Rule (CONTEXT.md): each token's attribution
divided evenly across the 6 nucleotides it spans. This is an explicit
analysis assumption, not a claim that the model perceives individual
nucleotides independently.

Both models' final attribution arrays are within-sequence L1-normalized
before any comparison - comparisons are relative-distribution based, never
raw magnitude - especially at the PAM (`NNGRRT`) and Seed (PAM-proximal
17-24bp) positions. Seed boundaries (17-24) reuse mismatch_profiling.py's
existing convention; PAM boundaries (25-30bp inclusive) are this dataset's
confirmed fixed sequence-construction alignment (36bp = ... + 8bp Seed at
17-24 + 6bp PAM at 25-30 + 5bp 3' flank at 31-35).
"""

import contextlib

import numpy as np
import torch
from ism_sweep import dispatch_in_batches
from mismatch_profiling import SEED_REGION_END, SEED_REGION_START
from model_a_wrapper import Model_A_Predictor
from model_b_xai_wrapper import Model_B_XAIPredictor

DEFAULT_IG_STEPS = 50

# Lower than ism_sweep.DEFAULT_PREDICT_BATCH_SIZE (256): unlike a single
# predict() forward pass, IG retains a full autograd graph per step across
# `steps` sequential backward passes, so it is far more memory-hungry per
# sequence - the same class of CUDA OOM ism_sweep.py's own batching guards
# against, just reached at a smaller batch size.
DEFAULT_IG_BATCH_SIZE = 32

CLS_TOKEN_COUNT = 1
TOKEN_NT_SPAN = (
    6  # 6-mer tokenizer: each non-CLS token covers 6 consecutive nucleotides
)

# PAM (NNGRRT), confirmed dataset alignment: 25-30 inclusive (25:31 as a slice).
PAM_REGION_START, PAM_REGION_END = 25, 31


@contextlib.contextmanager
def _frozen_parameters(module: torch.nn.Module):
    """Temporarily disable `requires_grad` on every parameter in `module`.

    Integrated Gradients only needs a gradient w.r.t. the interpolated
    *input* tensor - `backward()` still populates that regardless of
    whether `module`'s own weights require grad, since autograd tracks any
    graph touching a requires_grad leaf. Without this, backprop through
    Model B's ~500M-parameter NT encoder (unlike Model A's ~140K-parameter
    CNN+RNN) allocates a same-sized `.grad` buffer for every one of those
    weights at every interpolation step - pure waste that was enough to
    exhaust this machine's RAM once IG started replaying the full encoder
    (see `integrated_gradients_model_b_tokens`) instead of just its small
    classification head. Restores each parameter's original
    `requires_grad` afterward so other callers (e.g. `get_token_embeddings`'s
    gradient-flow assertion, Attention Rollout) see the model unchanged.
    """
    originally_required = [p.requires_grad for p in module.parameters()]
    for p in module.parameters():
        p.requires_grad_(False)
    try:
        yield
    finally:
        for p, required in zip(module.parameters(), originally_required):
            p.requires_grad_(required)


def _integrated_gradients(
    forward_fn, inputs: torch.Tensor, baseline: torch.Tensor, steps: int
) -> torch.Tensor:
    """Riemann-sum Integrated Gradients (Sundararajan et al., 2017).

    `forward_fn` must map a tensor shaped like `inputs` to a `(batch,)`
    scalar-per-sample output. Summing that output before `backward()` is
    safe because every sample's output depends only on its own slice of
    `inputs` (true for both Model A's per-sequence CNN+RNN forward pass and
    Model B's per-token classifier head) - the batch-summed gradient still
    equals each sample's own gradient, so one backward pass per step covers
    the whole batch instead of one per sample.

    Runs with cuDNN's fused RNN kernel disabled: that kernel only supports
    a backward pass when the forward ran in training mode, so Model A's
    eval-mode LSTM raises "cudnn RNN backward can only be called in
    training mode" on GPU otherwise (dropout=0 here makes train/eval mode
    otherwise identical, so this changes nothing except which kernel runs).
    Falls back to a slower, generic RNN backward that works in eval mode
    regardless - a no-op for Model B (no RNN involved) and on CPU (cuDNN
    never applies there), so this is safe to apply unconditionally.
    """
    diff = inputs - baseline
    accumulated_grad = torch.zeros_like(inputs)
    with torch.backends.cudnn.flags(enabled=False):
        for step in range(1, steps + 1):
            alpha = step / steps
            interpolated = (
                (baseline + alpha * diff).detach().clone().requires_grad_(True)
            )
            output = forward_fn(interpolated)
            output.sum().backward()
            accumulated_grad += interpolated.grad
    avg_grad = accumulated_grad / steps
    return (diff * avg_grad).detach()


def integrated_gradients_model_a(
    predictor: Model_A_Predictor,
    sequences: list[str],
    steps: int = DEFAULT_IG_STEPS,
    batch_size: int = DEFAULT_IG_BATCH_SIZE,
) -> np.ndarray:
    """Per-nucleotide IG attribution for Model A, shape `(batch, 36)`.

    Runs against Model A's one-hot input layer with an all-zero one-hot
    baseline. Since the baseline is zero and a one-hot input is zero
    everywhere except the observed base's channel, only that channel ever
    carries a nonzero `(x - baseline)` term - summing over the 4-channel
    axis collapses to the attribution of the base actually present at each
    position, discarding nothing. `sequences` is dispatched in
    `batch_size`-sized chunks (see `DEFAULT_IG_BATCH_SIZE`) to bound peak
    memory the same way `ism_sweep.py` bounds `predict()` calls.
    """

    def _attribute_chunk(chunk: list[str]) -> np.ndarray:
        inputs = predictor.encode_one_hot(chunk).to(predictor.device)
        baseline = torch.zeros_like(inputs)
        attributions = _integrated_gradients(predictor.model, inputs, baseline, steps)
        return attributions.sum(dim=-1).cpu().numpy()

    return dispatch_in_batches(_attribute_chunk, list(sequences), batch_size)


def integrated_gradients_model_b_tokens(
    xai_predictor: Model_B_XAIPredictor,
    sequences: list[str],
    steps: int = DEFAULT_IG_STEPS,
    batch_size: int = DEFAULT_IG_BATCH_SIZE,
) -> np.ndarray:
    """Raw per-token IG attribution for Model B, shape `(batch, 7, 1280)`, CLS included.

    Attributes against the `(batch, 7, 1280)` *pre-encoder* input-embedding
    tensor `Model_B_XAIPredictor.get_input_embeddings` exposes, using a
    fixed zero-embedding baseline of the same shape, backpropagating
    through `classify_from_input_embeddings` - the full encoder + head,
    not the head alone (see this module's docstring for why attributing
    against the *post-encoder* embedding instead would make every non-CLS
    token's attribution exactly zero by construction). `attention_mask` is
    fixed per sequence (padding structure, not a differentiable quantity)
    and passed through unchanged at every interpolation step. The
    embedding shape is validated before attribution runs; a mismatch
    raises here rather than silently attributing against a
    differently-shaped tensor. `get_input_embeddings` already asserts this
    on its own, but that assertion compiles out entirely under Python's
    `-O` flag - this check does not, so the guarantee holds independent of
    how the caller runs. `sequences` is dispatched in `batch_size`-sized
    chunks for the same reason `integrated_gradients_model_a` is. The
    actual backward loop runs under `_frozen_parameters` (see its
    docstring) so backprop through the full encoder doesn't allocate a
    `.grad` buffer for the encoder's own ~500M parameters at every step.
    """

    def _attribute_chunk(chunk: list[str]) -> np.ndarray:
        embeddings, attention_mask = xai_predictor.get_input_embeddings(chunk)
        embeddings = embeddings.detach()

        expected_shape = (
            len(chunk),
            Model_B_XAIPredictor.EXPECTED_SEQ_LEN,
            Model_B_XAIPredictor.EXPECTED_HIDDEN_DIM,
        )
        if tuple(embeddings.shape) != expected_shape:
            raise ValueError(
                f"Model B embedding shape {tuple(embeddings.shape)} does not match "
                f"expected {expected_shape}; refusing to attribute against a "
                "differently-shaped tensor."
            )

        baseline = torch.zeros_like(embeddings)

        def _forward(interpolated: torch.Tensor) -> torch.Tensor:
            return xai_predictor.classify_from_input_embeddings(
                interpolated, attention_mask
            )

        # Frozen only around the actual backward loop, not `get_input_embeddings`
        # above - that's a single forward pass (never calls `.backward()`, so
        # frozen-or-not costs nothing there) whose own internal assertion
        # requires a normal, grad-enabled parameter state to pass.
        with _frozen_parameters(xai_predictor.nt_model):
            attributions = _integrated_gradients(_forward, embeddings, baseline, steps)
        return attributions.cpu().numpy()

    return dispatch_in_batches(_attribute_chunk, list(sequences), batch_size)


def project_model_b_attributions_to_nucleotides(
    token_attributions: np.ndarray,
) -> np.ndarray:
    """Deterministic Projection Rule: 6-mer token attribution -> 36bp nucleotide attribution.

    `token_attributions` is `(batch, 7, 1280)` (CLS + 6 tokens, hidden
    dimension not yet collapsed). Collapses the hidden dimension by summing
    (mirroring Model A's one-hot-channel collapse), drops the CLS token,
    then divides each of the 6 remaining tokens' attribution evenly across
    the 6 nucleotides it spans - an explicit analysis assumption (see
    CONTEXT.md's Deterministic Projection Rule), not evidence the model
    perceives individual nucleotides independently. Conservation holds:
    each nucleotide block sums back to exactly its source token's
    attribution.
    """
    _batch, n_tokens, _hidden = token_attributions.shape
    if n_tokens != Model_B_XAIPredictor.EXPECTED_SEQ_LEN:
        raise ValueError(
            f"Expected {Model_B_XAIPredictor.EXPECTED_SEQ_LEN} tokens, got {n_tokens}"
        )

    per_token = token_attributions.sum(axis=-1)  # (batch, 7)
    per_kmer_token = per_token[:, CLS_TOKEN_COUNT:]  # drop CLS -> (batch, 6)

    return (
        np.repeat(per_kmer_token, TOKEN_NT_SPAN, axis=1) / TOKEN_NT_SPAN
    )  # (batch, 36)


def l1_normalize(attribution: np.ndarray) -> np.ndarray:
    """Within-sequence L1 normalization: each row's absolute values sum to 1.

    Model A and Model B's attribution magnitudes are on unrelated scales -
    comparisons are only meaningful as relative distributions (per
    plan_realize.md Step 2), never raw magnitude. All-zero rows stay
    all-zero (dividing by zero L1 mass would otherwise produce NaN) since
    there is no distribution to normalize.
    """
    l1_mass = np.abs(attribution).sum(axis=-1, keepdims=True)
    safe_mass = np.where(l1_mass == 0, 1.0, l1_mass)
    return attribution / safe_mass


def region_attribution_share(
    normalized_attribution: np.ndarray, start: int, end: int
) -> np.ndarray:
    """Fraction of a sequence's (already L1-normalized) attribution mass in `[start, end)`."""
    return np.abs(normalized_attribution[:, start:end]).sum(axis=-1)


def run_integrated_gradients(
    sequences: list[str],
    predictor_a: Model_A_Predictor,
    xai_predictor_b: Model_B_XAIPredictor,
    steps: int = DEFAULT_IG_STEPS,
    batch_size: int = DEFAULT_IG_BATCH_SIZE,
) -> dict:
    """Run IG for both models and the PAM/Seed relative-distribution comparison.

    Returns raw and L1-normalized per-nucleotide attribution arrays for each
    model, Model B's pre-projection per-token attribution (CLS dropped), and
    each model's PAM (25-30bp) / Seed (17-24bp) attribution-mass share - the
    relative-distribution comparison issue #15's AC4 calls for.
    """
    ig_a = integrated_gradients_model_a(
        predictor_a, sequences, steps=steps, batch_size=batch_size
    )
    ig_a_norm = l1_normalize(ig_a)

    ig_b_tokens = integrated_gradients_model_b_tokens(
        xai_predictor_b, sequences, steps=steps, batch_size=batch_size
    )
    ig_b = project_model_b_attributions_to_nucleotides(ig_b_tokens)
    ig_b_norm = l1_normalize(ig_b)

    return {
        "ig_model_a": ig_a,
        "ig_model_a_normalized": ig_a_norm,
        "ig_model_b_tokens": ig_b_tokens[:, CLS_TOKEN_COUNT:, :].sum(axis=-1),
        "ig_model_b": ig_b,
        "ig_model_b_normalized": ig_b_norm,
        "pam_share_model_a": region_attribution_share(
            ig_a_norm, PAM_REGION_START, PAM_REGION_END
        ),
        "pam_share_model_b": region_attribution_share(
            ig_b_norm, PAM_REGION_START, PAM_REGION_END
        ),
        "seed_share_model_a": region_attribution_share(
            ig_a_norm, SEED_REGION_START, SEED_REGION_END
        ),
        "seed_share_model_b": region_attribution_share(
            ig_b_norm, SEED_REGION_START, SEED_REGION_END
        ),
    }


In [ ]:
%%writefile test_integrated_gradients.py
import numpy as np
import pytest
import torch
from integrated_gradients import (
    DEFAULT_IG_STEPS,
    PAM_REGION_END,
    PAM_REGION_START,
    _integrated_gradients,
    integrated_gradients_model_a,
    integrated_gradients_model_b_tokens,
    l1_normalize,
    project_model_b_attributions_to_nucleotides,
    region_attribution_share,
    run_integrated_gradients,
)
from mismatch_profiling import SEED_REGION_END, SEED_REGION_START
from model_a_wrapper import Model_A_Predictor
from model_b_xai_wrapper import Model_B_XAIPredictor

NT_MODEL_DIR = "./NT_sacas9_fintuned_model"

# Same fixture sequences as test_model_b_xai_wrapper.py (sample_id 0-2).
TESTSET_SEQUENCES = [
    "CCCGGCTGACCTGCCCAAGCTGGTGGAGGGGCTGAA",
    "TTGAAGGAAGGGAATCCAGGTGTGTAAGGGTCACCT",
    "GGAGGAGGAAGAAGAACTGGAAGAGGTGGAAGACCT",
]


# ---------------------------------------------------------------------------
# _integrated_gradients: the completeness axiom (Sundararajan et al., 2017)
# holds exactly for a linear function regardless of step count, since the
# gradient is constant along the whole straight-line path.
# ---------------------------------------------------------------------------


def test_integrated_gradients_satisfies_completeness_for_a_linear_function():
    torch.manual_seed(0)
    weights = torch.randn(5)

    def linear_fn(x):
        return x @ weights

    inputs = torch.randn(4, 5)
    baseline = torch.zeros(4, 5)

    attributions = _integrated_gradients(linear_fn, inputs, baseline, steps=1)

    expected_delta = linear_fn(inputs) - linear_fn(baseline)
    torch.testing.assert_close(
        attributions.sum(dim=-1), expected_delta, atol=1e-5, rtol=1e-5
    )


def test_integrated_gradients_completeness_is_independent_of_step_count():
    torch.manual_seed(1)
    weights = torch.randn(3)

    def linear_fn(x):
        return x @ weights

    inputs = torch.randn(2, 3)
    baseline = torch.randn(2, 3)

    few_steps = _integrated_gradients(linear_fn, inputs, baseline, steps=1)
    many_steps = _integrated_gradients(linear_fn, inputs, baseline, steps=50)

    torch.testing.assert_close(few_steps, many_steps, atol=1e-5, rtol=1e-5)


def test_integrated_gradients_is_zero_when_input_equals_baseline():
    weights = torch.randn(6)

    def linear_fn(x):
        return x @ weights

    inputs = torch.randn(3, 6)

    attributions = _integrated_gradients(linear_fn, inputs, inputs.clone(), steps=10)

    torch.testing.assert_close(attributions, torch.zeros_like(attributions))


# ---------------------------------------------------------------------------
# Model A: one-hot input layer, all-zero one-hot baseline
# ---------------------------------------------------------------------------


@pytest.fixture(scope="module")
def model_a_predictor():
    # Random-initialized weights: IG's algorithmic correctness (shape,
    # completeness) doesn't depend on trained weights, and skipping the
    # weight file keeps this test module fast and independent of it.
    return Model_A_Predictor(weight_path=None)


def test_integrated_gradients_model_a_shape(model_a_predictor):
    attributions = integrated_gradients_model_a(model_a_predictor, TESTSET_SEQUENCES)

    assert attributions.shape == (len(TESTSET_SEQUENCES), 36)


def test_integrated_gradients_model_a_approximately_satisfies_completeness(
    model_a_predictor,
):
    sequences = TESTSET_SEQUENCES[:1]
    attributions = integrated_gradients_model_a(model_a_predictor, sequences, steps=200)

    inputs = model_a_predictor.encode_one_hot(sequences).to(model_a_predictor.device)
    baseline = torch.zeros_like(inputs)
    with torch.no_grad():
        expected_delta = (
            (model_a_predictor.model(inputs) - model_a_predictor.model(baseline))
            .cpu()
            .numpy()
        )

    np.testing.assert_allclose(attributions.sum(axis=-1), expected_delta, atol=1e-3)


def test_integrated_gradients_model_a_zero_sequence_yields_zero_attribution(
    model_a_predictor,
):
    # A sequence whose one-hot encoding *is* the baseline (all-zero) has no
    # (x - baseline) term anywhere, so attribution must be exactly zero -
    # this exercises the actual encoder path, unlike the pure-tensor tests
    # above which build the zero input directly. Must live on the model's
    # own device (cuda on a GPU runtime) - a bare CPU tensor only happened
    # to work locally because this machine's device is always CPU anyway.
    inputs = torch.zeros(1, 36, 4, device=model_a_predictor.device)
    baseline = torch.zeros_like(inputs)

    attributions = _integrated_gradients(
        model_a_predictor.model, inputs, baseline, steps=5
    )

    torch.testing.assert_close(attributions, torch.zeros_like(attributions))


def test_integrated_gradients_model_a_batch_size_does_not_change_values(
    model_a_predictor,
):
    one_shot = integrated_gradients_model_a(
        model_a_predictor, TESTSET_SEQUENCES, steps=5, batch_size=10_000
    )
    batched = integrated_gradients_model_a(
        model_a_predictor, TESTSET_SEQUENCES, steps=5, batch_size=1
    )

    np.testing.assert_allclose(one_shot, batched, atol=1e-6)


# ---------------------------------------------------------------------------
# Model B: (batch, 7, 1280) embedding tensor, zero-embedding baseline
# ---------------------------------------------------------------------------


class _WrongShapeXAIPredictor:
    """Stand-in exposing only `get_input_embeddings`, returning a
    deliberately mis-shaped tensor without raising - proves
    `integrated_gradients_model_b_tokens`'s own explicit shape check fires
    independently of `Model_B_XAIPredictor.get_input_embeddings`'s internal
    `assert` (which would otherwise always fire first against a real
    predictor, per `test_integrated_gradients_model_b_tokens_malformed_input_raises`
    below, and which compiles out entirely under Python's `-O` flag).
    """

    def get_input_embeddings(
        self, sequences: list[str]
    ) -> tuple[torch.Tensor, torch.Tensor]:
        batch = len(sequences)
        return (
            torch.zeros(batch, 5, 1280, requires_grad=True),
            torch.ones(batch, 5),
        )


@pytest.fixture(scope="module")
def xai_predictor():
    return Model_B_XAIPredictor(nt_model_dir=NT_MODEL_DIR)


def test_integrated_gradients_model_b_tokens_shape(xai_predictor):
    attributions = integrated_gradients_model_b_tokens(
        xai_predictor, TESTSET_SEQUENCES, steps=5
    )

    assert attributions.shape == (len(TESTSET_SEQUENCES), 7, 1280)


def test_integrated_gradients_model_b_tokens_malformed_input_raises(xai_predictor):
    # 12bp sequence tokenizes to 3 tokens (<cls> + two 6-mers), not 7 - see
    # test_model_b_xai_wrapper.py's identical malformed-input case. Against
    # a real predictor, `get_token_embeddings`'s own internal assert always
    # fires before our explicit check ever runs; that check is covered
    # separately below via `_WrongShapeXAIPredictor`.
    with pytest.raises(AssertionError):
        integrated_gradients_model_b_tokens(xai_predictor, ["ATCGATCGATCG"], steps=5)


def test_integrated_gradients_model_b_tokens_raises_on_mismatched_embedding_shape():
    with pytest.raises(ValueError, match="does not match expected"):
        integrated_gradients_model_b_tokens(
            _WrongShapeXAIPredictor(), TESTSET_SEQUENCES, steps=5
        )


def test_integrated_gradients_model_b_tokens_batch_size_does_not_change_values(
    xai_predictor,
):
    one_shot = integrated_gradients_model_b_tokens(
        xai_predictor, TESTSET_SEQUENCES, steps=5, batch_size=10_000
    )
    batched = integrated_gradients_model_b_tokens(
        xai_predictor, TESTSET_SEQUENCES, steps=5, batch_size=1
    )

    np.testing.assert_allclose(one_shot, batched, atol=1e-5)


def test_integrated_gradients_model_b_tokens_approximately_satisfies_completeness(
    xai_predictor,
):
    sequences = TESTSET_SEQUENCES[:1]
    attributions = integrated_gradients_model_b_tokens(
        xai_predictor, sequences, steps=100
    )

    embeddings, attention_mask = xai_predictor.get_input_embeddings(sequences)
    embeddings = embeddings.detach()
    baseline = torch.zeros_like(embeddings)
    with torch.no_grad():
        expected_delta = (
            (
                xai_predictor.classify_from_input_embeddings(embeddings, attention_mask)
                - xai_predictor.classify_from_input_embeddings(baseline, attention_mask)
            )
            .cpu()
            .numpy()
        )

    # Much looser tolerance than Model A's completeness check: this path
    # curves through 24 full transformer layers' worth of nonlinearities
    # (zero embedding -> a real, large-magnitude NT embedding), not just a
    # 2-layer head, so a finite-step Riemann sum leaves a larger residual
    # even at steps=100 (empirically ~9% relative here).
    np.testing.assert_allclose(
        attributions.sum(axis=(-1, -2)), expected_delta, atol=0.06, rtol=0.15
    )


# ---------------------------------------------------------------------------
# Deterministic Projection Rule
# ---------------------------------------------------------------------------


def test_project_model_b_attributions_conserves_mass_per_token_block():
    rng = np.random.default_rng(0)
    token_attributions = rng.normal(size=(2, 7, 1280)).astype(np.float32)

    nucleotide_attr = project_model_b_attributions_to_nucleotides(token_attributions)

    assert nucleotide_attr.shape == (2, 36)
    per_token = token_attributions.sum(axis=-1)[:, 1:]  # drop CLS -> (2, 6)
    for token_idx in range(6):
        block = nucleotide_attr[:, token_idx * 6 : (token_idx + 1) * 6]
        np.testing.assert_allclose(
            block.sum(axis=-1), per_token[:, token_idx], atol=1e-5
        )


def test_project_model_b_attributions_divides_evenly_within_a_token_block():
    token_attributions = np.zeros((1, 7, 4), dtype=np.float32)
    token_attributions[0, 1, :] = [6.0, 0.0, 0.0, 0.0]  # first 6-mer token, sum = 6

    nucleotide_attr = project_model_b_attributions_to_nucleotides(token_attributions)

    np.testing.assert_allclose(nucleotide_attr[0, :6], np.full(6, 1.0))
    np.testing.assert_allclose(nucleotide_attr[0, 6:], np.zeros(30))


def test_project_model_b_attributions_rejects_wrong_token_count():
    token_attributions = np.zeros((1, 5, 1280), dtype=np.float32)

    with pytest.raises(ValueError):
        project_model_b_attributions_to_nucleotides(token_attributions)


# ---------------------------------------------------------------------------
# L1 normalization + region share
# ---------------------------------------------------------------------------


def test_l1_normalize_rows_sum_to_one_in_absolute_value():
    rng = np.random.default_rng(2)
    attribution = rng.normal(size=(5, 36)).astype(np.float32)

    normalized = l1_normalize(attribution)

    np.testing.assert_allclose(np.abs(normalized).sum(axis=-1), np.ones(5), atol=1e-6)


def test_l1_normalize_preserves_sign_and_relative_proportion():
    attribution = np.array([[2.0, -2.0, 4.0, 0.0]])

    normalized = l1_normalize(attribution)

    np.testing.assert_allclose(normalized, [[0.25, -0.25, 0.5, 0.0]])


def test_l1_normalize_leaves_all_zero_rows_as_zero():
    attribution = np.zeros((1, 36))

    normalized = l1_normalize(attribution)

    np.testing.assert_allclose(normalized, np.zeros((1, 36)))
    assert not np.isnan(normalized).any()


def test_region_attribution_share_sums_absolute_mass_in_range():
    normalized = np.zeros((1, 36))
    normalized[0, PAM_REGION_START:PAM_REGION_END] = [0.1, -0.1, 0.1, -0.1, 0.1, -0.1]

    pam_share = region_attribution_share(normalized, PAM_REGION_START, PAM_REGION_END)

    np.testing.assert_allclose(pam_share, [0.6])


def test_region_attribution_share_is_bounded_by_total_l1_mass():
    rng = np.random.default_rng(3)
    attribution = rng.normal(size=(4, 36)).astype(np.float32)
    normalized = l1_normalize(attribution)

    pam_share = region_attribution_share(normalized, PAM_REGION_START, PAM_REGION_END)
    seed_share = region_attribution_share(
        normalized, SEED_REGION_START, SEED_REGION_END
    )

    assert np.all(pam_share >= 0.0) and np.all(pam_share <= 1.0)
    assert np.all(seed_share >= 0.0) and np.all(seed_share <= 1.0)
    assert np.all(pam_share + seed_share <= 1.0 + 1e-6)  # regions don't overlap


# ---------------------------------------------------------------------------
# Full bundle
# ---------------------------------------------------------------------------


def test_run_integrated_gradients_returns_the_full_comparison_bundle(
    model_a_predictor, xai_predictor
):
    result = run_integrated_gradients(
        TESTSET_SEQUENCES, model_a_predictor, xai_predictor, steps=5
    )

    assert set(result.keys()) == {
        "ig_model_a",
        "ig_model_a_normalized",
        "ig_model_b_tokens",
        "ig_model_b",
        "ig_model_b_normalized",
        "pam_share_model_a",
        "pam_share_model_b",
        "seed_share_model_a",
        "seed_share_model_b",
    }

    n = len(TESTSET_SEQUENCES)
    assert result["ig_model_a"].shape == (n, 36)
    assert result["ig_model_a_normalized"].shape == (n, 36)
    assert result["ig_model_b_tokens"].shape == (n, 6)
    assert result["ig_model_b"].shape == (n, 36)
    assert result["ig_model_b_normalized"].shape == (n, 36)
    for key in (
        "pam_share_model_a",
        "pam_share_model_b",
        "seed_share_model_a",
        "seed_share_model_b",
    ):
        assert result[key].shape == (n,)
        assert np.all(result[key] >= 0.0) and np.all(result[key] <= 1.0)


def test_default_ig_steps_is_a_positive_int():
    assert isinstance(DEFAULT_IG_STEPS, int)
    assert DEFAULT_IG_STEPS > 0


## 5. Confirm GPU is available

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected - Runtime > Change runtime type > GPU")

## 6. Run the full Model B Integrated Gradients test suite

This is the exact suite that caught the original all-zero bug
(`test_integrated_gradients_model_b_tokens_approximately_satisfies_completeness`
failed against the head-only path) and every other IG test alongside it -
Model A's tests are cheap (a random-initialized, ~140K-parameter dummy
model - no weight file needed) so the whole file runs together.


In [ ]:
!python -m pytest test_integrated_gradients.py -v

## 7. Direct before/after check: real, non-zero per-token attribution

Beyond the test suite passing, this makes the fix tangible: real Testset
sequences now get a genuine, non-degenerate per-nucleotide attribution
distribution for Model B, not 36 zeros.


In [ ]:
import numpy as np
from integrated_gradients import (
    integrated_gradients_model_b_tokens,
    l1_normalize,
    project_model_b_attributions_to_nucleotides,
)
from model_b_xai_wrapper import Model_B_XAIPredictor

# Same 3 fixture sequences test_integrated_gradients.py / test_model_b_xai_wrapper.py use.
sequences = [
    "CCCGGCTGACCTGCCCAAGCTGGTGGAGGGGCTGAA",
    "TTGAAGGAAGGGAATCCAGGTGTGTAAGGGTCACCT",
    "GGAGGAGGAAGAAGAACTGGAAGAGGTGGAAGACCT",
]

xai_predictor = Model_B_XAIPredictor(nt_model_dir="./NT_sacas9_fintuned_model")

ig_tokens = integrated_gradients_model_b_tokens(xai_predictor, sequences, steps=50)
ig_nt = project_model_b_attributions_to_nucleotides(ig_tokens)
ig_nt_norm = l1_normalize(ig_nt)

per_token_sum = ig_tokens.sum(axis=-1)  # (3, 7): CLS + 6 tokens
print("Per-token attribution sum (col 0 = CLS, cols 1-6 = the 6 real 6-mer tokens):")
print(per_token_sum)
print()
print(
    "Non-CLS tokens all exactly zero would mean the old bug is still present.",
    "Nonzero values below confirm the fix:",
)
print("Any non-CLS token nonzero:", bool(np.any(per_token_sum[:, 1:] != 0)))
print()
print(
    "L1-normalized per-nucleotide attribution, sequence 0 (36 values, abs sums to 1):"
)
print(np.round(ig_nt_norm[0], 4))
print("Row abs-sum (should be 1.0):", np.abs(ig_nt_norm[0]).sum())

## 8. What a green run confirms

- All of `test_integrated_gradients.py` passes, including the completeness
  check that previously failed with the old tolerance/path.
- Step 6's printed per-token sums are nonzero for tokens 1-6 (the real
  nucleotide-bearing tokens), not just token 0 (CLS) - the old bug produced
  exactly zero for columns 1-6 on every sample, every time.
- The L1-normalized attribution row is a genuine, non-degenerate
  distribution (not 36 zeros).

If you want the corrected values folded back into
`model_analysis_summary.json`, re-run `step5_export.py`'s Case Study
Integrated Gradients step here (Model A + Model B_Predictor/XGBoost would
also need to be loaded - see `compute_case_study_ig` in `step5_export.py`)
and copy the two output files back to the local `xai_test/` root, the same
way `colab_shap.ipynb`'s results were folded in.
